# HW 2: Discrete Heterogeneity

Part 1: Re-estimate the multinomial logit brand choice model with the last brand purchased variable on the Yogurt data, but using a 2 type discrete heterogeneity distribution.  For now put the heterogeneity only on the intercepts. The last assignment had you form an individual specific likelihood function that took an NT by 1 vector and collapsed it to N by 1. For this assignment, first form a likelihood function that is NT by M. Then take the products across T within the first dimension to collapse it into an N by M matrix with the likelihood for each individual if they were each type. Then, take the weighted average of the types over the 2nd dimension to form the individual specific likelihood function which is N by 1.

Part 2: Now try allowing all of the parameters to be type specific (still use 2 types).  Also, try allowing for 4 types but restricting types to vary only by intercepts.

Part 3: Numerically solve for cross price elasticities in the homogenous multinomial logit model and the two type finite support heterogeneity model.  To do so, simulate a small change in the price of brand 1.  Also try it by simulating a small price change for brand 2. Compare elasticities between the homogenous and heterogenous models.

In [1]:
# Packages
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.special import logsumexp
from scipy.stats import norm
from statsmodels.tools.numdiff import approx_hess
import multiprocessing
from concurrent.futures import ProcessPoolExecutor, as_completed
import time

In [2]:
# Import data
df = pd.read_csv("../data/YogurtLong.txt", sep="\t")
df.head()

,hh,qty,exp,nopurch,b1,b2,b3,b4,p1,p2,p3,p4,f1,f2,f3,f4,tripnum
0,1,2,40.900002,0,0,0,0,1,0.11,0.08,0.06,0.08,0,0,0,0,1
1,1,0,8.830000,1,0,0,0,0,0.11,0.09,0.05,0.08,0,0,0,0,2
2,1,0,3.880000,1,0,0,0,0,0.13,0.09,0.05,0.08,0,0,0,0,3
3,1,0,0.780000,1,0,0,0,0,0.13,0.09,0.05,0.08,0,0,0,0,4
4,1,0,39.240002,1,0,0,0,0,0.13,0.09,0.05,0.08,0,0,0,0,5


In [3]:
# Convert df to long (each row is a hh-trip-brand choice occasion)
df_long = (
    pd.wide_to_long(
        df,
        stubnames=["b", "p", "f"],
        i=["hh", "tripnum"],
        j="brand",
        suffix=r"\d+"
    )
    .reset_index()
    .sort_values(["hh", "tripnum", "brand"])
)
df_long = pd.get_dummies(df_long, columns=["brand"], prefix="brand", dtype=int)

# Create prev_purch variable
df_long["brand_num"] = df_long[["brand_1", "brand_2", "brand_3", "brand_4"]].idxmax(axis=1).str.replace("brand_", "").astype(int)
trip_choice = (
    df_long.loc[df_long["b"] == 1, ["hh", "tripnum", "brand_num"]]
    .rename(columns={"brand_num": "chosen_brand"})
)
trip_hist = (
    df_long[["hh", "tripnum"]]
    .drop_duplicates()
    .sort_values(["hh", "tripnum"])
    .merge(trip_choice, on=["hh", "tripnum"], how="left")
)
trip_hist["prev_purch"] = (
    trip_hist.groupby("hh")["chosen_brand"]
    .transform(lambda x: x.ffill().shift(1))
    .fillna(0)
    .astype(int)
)
df_long = df_long.merge(
    trip_hist[["hh", "tripnum", "prev_purch"]],
    on=["hh", "tripnum"],
    how="left",
)

df_long.head()

,hh,tripnum,exp,nopurch,qty,b,p,f,brand_1,brand_2,brand_3,brand_4,brand_num,prev_purch
0,1,1,40.900002,0,2,0,0.11,0,1,0,0,0,1,0
1,1,1,40.900002,0,2,0,0.08,0,0,1,0,0,2,0
2,1,1,40.900002,0,2,0,0.06,0,0,0,1,0,3,0
3,1,1,40.900002,0,2,1,0.08,0,0,0,0,1,4,0
4,1,2,8.830000,1,0,0,0.11,0,1,0,0,0,1,4


## I. Discrete latent-class heterogeneity implementation

Suppose there are $J$ products and $C$ latent classes. We let brand intercepts $\beta_{0cj}$ ($j=1,\ldots,J$) be class-specific and $\tilde{k}$ other covariates $\beta_1, \ldots, \beta_{\tilde{k}}$ be common across classes. Then, class $c$ has the covariate vector $\vec\beta_c = [\beta_{0c1}, \ldots, \beta_{0cJ}, \beta_1, \cdots, \beta_{\tilde{k}}]^\intercal \in \mathbb{R}^{k}$ where $k=J+\tilde{k}$ is the number of parameters in each covariate vector.

In the following implementation, I create a matrix $B \in \mathbb{R}^{k \times C}$ that stores the covariate vector of class $c$ as its $c$th column (note that $B$ stores all the covariate coefficients to be estimated):
$$
B = \begin{bmatrix} \vec\beta_1 & \cdots & \vec\beta_C \end{bmatrix} = 
\begin{bmatrix} 
\beta_{011} & \cdots & \beta_{0C1} \\
\vdots & \ddots & \vdots \\
\beta_{01J} & \cdots & \beta_{0CJ} \\
\beta_1 & \cdots & \beta_1 \\ 
\vdots & \ddots & \vdots \\
\beta_{\tilde{k}} & \cdots & \beta_{\tilde{k}} \\ 
\end{bmatrix}
$$
Recall that $X \in \mathbb{R}^{NTJ \times k}$ is the design matrix ($NT$ is the number of household-trips made, or equivalently, the number of choice occasions observed). Then $V = XB \in \mathbb{R}^{NTJ \times C}$ stores the utility $v_{itj\mid c}$ of household $i$ in trip $t$ choosing product $j$ if they were of class $c$:
$$
V = \begin{bmatrix}
v_{111\mid 1} & \cdots & v_{111\mid C} \\
\vdots & \ddots & \vdots \\
v_{11J\mid 1} & \cdots & v_{11J\mid C} \\
v_{121\mid 1} & \cdots & v_{121\mid C} \\
\vdots & \ddots & \vdots \\
v_{itj\mid 1} & \cdots & v_{itj\mid C} \\
\vdots & \ddots & \vdots \\
v_{NTJ\mid 1} & \cdots & v_{NTJ\mid C} \\
\end{bmatrix}
$$
We desire an $N \times C$ matrix that contains the likelihoods of each household choices as if they were of each class. To do so, we first reshape $V$ to shape $NT \times J \times C$ and collapse across the second dimension using $1+\sum_{j=1}^J \exp(v_{itj})$. Call the resulting matrix $L_\text{den} \in \mathbb{R}^{NT \times C}$—this is the denominator of the MNL formula. Letting $Y \in \mathbb{R}^{NTJ}$ be the binary vector of whether product $j$ was chosen for each row $itj$, form $\tilde{Y} \in \mathbb{R}^{NTJ \times C}$ that simply copies $Y$ in its $C$ columns. Then, $V \odot \tilde{Y}$ (element-wise matrix multiplication) and reshaping to shape $NT \times J \times C$ before collapsing across the second dimension with a regular sum returns a matrix $V_\text{chosen} \in \mathbb{R}^{NT \times C}$ containing the utility obtained by household $i$ in trip $t$ due to their observed choice, if they were of class $c$. Exponentiating each value gives us $L_\text{num} \in \mathbb{R}^{NT \times C}$—the numerator of the MNL formula. Element-wise division of $L_\text{num}$ over $L_\text{den}$ creates $L^{(3)} \in \mathbb{R}^{NT \times C}$, where $L^{(3)}_{itc}$ is the probability of observing household $i$ choose the product they chose in trip $t$, if they were of class $c$. Taking the product over all the trips $t$ made by household $i$ gives $L^{(2)} \in \mathbb{R}^{N \times C}$: $L^{(2)}_{hc} = \prod_t L^{(3)}_{itc}$.

Now let $\mu=[\mu_1, \ldots, \mu_C]$ be the proportions of each class ($\mu_c \geq 0, \sum_c \mu_c = 1$). To avoid the non-negativity and normalization constraints, we estimate $\tilde\mu = \{\tilde\mu_1, \ldots, \tilde\mu_C\}$ where $\tilde\mu_c \in \mathbb{R}$ for $c = 2, \ldots, C$ and $\tilde\mu_1=0$. Then, we recover $\mu_c = \frac{\exp(\tilde\mu_c)}{\sum_c \exp(\tilde\mu_c)}$. To obtain the likelihoods of each household, we compute $L^{(1)} = L^{(2)}\mu \in \mathbb{R}^{N}$, which takes a $\mu$-weighted sum over classes.

Finally, we use maximum likelihood estimation on $\sum_{i=1}^N \log L^{(1)}_i$ to obtain parameter estimates for $B$ and $\mu$.

For numerical stability, the log-likelihood is computed in log-space. The full expression for the log-likelihood is given by:
$$
\sum_{i=1}^N \log(L^{(1)}) = \sum_{i=1}^N \mathrm{logsumexp}_c\left( \tilde\mu - \mathrm{logsumexp}_c(\tilde\mu) + \sum_t \left( V_\text{chosen} - \mathrm{logsumexp}_j([\mathbf{0}, V]) \right) \right)
$$
where the $\mathrm{logsumexp}_i(x)$ of a vector $x = ([x_i]_{i=1}^n)^\intercal$ is the expression $\log(\sum_{i=1}^n \exp(x_i))$. Under the hood, the `logsumexp` function applies a numerical stability trick to avoid overflow issues. Further note that in the above expression, we abuse notation to let $\tilde\mu - \mathrm{logsumexp}_c(\tilde\mu)$ be an $N \times C$ matrix where the length-$C$ vector is copied $N$ times.

In [19]:
def _unpack_theta(theta, k, n_classes, n_heterogenous):
    """
    Unpack theta into B (k x C) and free mu parameters (length C-1).

    Theta ordering: [heterogeneous coefs, common coefs, mu_free].
    Heterogeneous coefficients are column-stacked by class.
    """
    theta = np.asarray(theta).reshape(-1)
    n_hetero = int(n_heterogenous)
    n_common = k - n_hetero
    hetero_size = n_hetero * n_classes
    beta_size = hetero_size + n_common

    B = np.zeros((k, n_classes), dtype=float)

    if n_hetero > 0:
        B_hetero = theta[:hetero_size].reshape(n_hetero, n_classes, order="F")
        B[:n_hetero, :] = B_hetero

    if n_common > 0:
        B_common = theta[hetero_size:beta_size].reshape(n_common)
        B[n_hetero:, :] = B_common[:, None]

    mu_free = theta[beta_size:].reshape(-1)
    return B, mu_free


def _negloglik_hh(theta, X, Y, hh_ids, n_alts, n_classes, n_heterogenous):
    """
    Negative log-likelihood for finite-mixture MNL with outside option,
    computed at the household level (vector of length n_households).
    """
    X = np.asarray(X)
    Y = np.asarray(Y).reshape(-1)
    hh_ids = np.asarray(hh_ids).reshape(-1)

    n, k = X.shape
    n_choices = n // n_alts  # NT
    B, mu_free = _unpack_theta(theta, k, n_classes, n_heterogenous=n_heterogenous)

    # L1: likelihood per hh-trip choice occasion, for each class
    V = (X @ B).reshape(n_choices, n_alts, n_classes, order="C")            # Utility tensor: (NT, J, C)
    V0 = np.concatenate([np.zeros((n_choices, 1, n_classes)), V], axis=1)   # add outside option with utility 0
    Y1 = Y.reshape(n_choices, n_alts, 1, order="C")
    chosen_v = np.multiply(Y1, V).sum(axis=1)   # (NT, C): Chosen utility per (choice occasion, class)
    log_L3 = chosen_v - logsumexp(V0, axis=1)

    # L2: likelihood per household, for each class
    hh_trip = hh_ids.reshape(n_choices, n_alts, order="C")[:, 0]
    _, hh_index = np.unique(hh_trip, return_inverse=True)
    log_L2 = np.column_stack([
        np.bincount(hh_index, weights=log_L3[:, c])
        for c in range(n_classes)
    ])

    # L1: likelihood per household, integrating over classes w.r.t. mu
    mu_raw = np.concatenate(([0.0], mu_free)) if n_classes > 1 else np.array([0.0])
    prop_class = mu_raw - logsumexp(mu_raw)     # back out class proportions
    log_L1 = logsumexp(log_L2 + prop_class[None, :], axis=1)

    return -log_L1


def _negloglik(theta, X, Y, hh_ids, n_alts, n_classes, n_heterogenous):
    """Summed negative log-likelihood"""
    log_L1 = _negloglik_hh(theta, X, Y, hh_ids, n_alts, n_classes, n_heterogenous=n_heterogenous)
    return np.sum(log_L1)


def _score_matrix(theta, X, Y, hh_ids, n_alts, n_classes, n_heterogenous, eps=1e-6):
    """
    Numerical score matrix of household-level negative log-likelihood contributions.
    Aggregation at the hh_ids level (so this is cluster-robust to household correlation).
    Shape: (n_households, n_params).
    """
    theta = np.asarray(theta).reshape(-1)
    p = theta.size

    base = _negloglik_hh(theta, X, Y, hh_ids, n_alts, n_classes, n_heterogenous=n_heterogenous)
    n_households = base.size
    S = np.empty((n_households, p), dtype=float)

    # Numerical differentiation with central difference approximation
    for k in range(p):
        step = eps * max(1.0, abs(theta[k]))
        theta_plus = theta.copy()
        theta_minus = theta.copy()
        theta_plus[k] += step
        theta_minus[k] -= step

        f_plus = _negloglik_hh(theta_plus, X, Y, hh_ids, n_alts, n_classes, n_heterogenous=n_heterogenous)
        f_minus = _negloglik_hh(theta_minus, X, Y, hh_ids, n_alts, n_classes, n_heterogenous=n_heterogenous)
        S[:, k] = (f_plus - f_minus) / (2.0 * step)

    return S


def _print_mle_output(output, covariate_vars, heterogenous_vars, robust_se, individual_var):
    """Prints MLE results in aligned tables, separated by class."""
    B_hat = np.asarray(output['opt_beta'])
    n_classes = B_hat.shape[1]
    k = B_hat.shape[0]

    se_theta = np.asarray(output['se_theta'])
    ci_theta = np.asarray(output['ci_theta'])

    if heterogenous_vars is None:
        n_hetero = k
    else:
        n_hetero = len(heterogenous_vars)

    n_common = k - n_hetero
    hetero_size = n_hetero * n_classes

    se_beta = np.zeros((k, n_classes), dtype=float)
    ci_beta = np.zeros((k, n_classes, 2), dtype=float)

    if n_hetero > 0:
        se_beta[:n_hetero, :] = se_theta[:hetero_size].reshape(n_hetero, n_classes, order="F")
        ci_beta[:n_hetero, :, :] = ci_theta[:hetero_size].reshape(n_hetero, n_classes, 2, order="F")

    if n_common > 0:
        se_common = se_theta[hetero_size:hetero_size + n_common]
        ci_common = ci_theta[hetero_size:hetero_size + n_common]
        se_beta[n_hetero:, :] = se_common[:, None]
        ci_beta[n_hetero:, :, :] = ci_common[:, None, :]

    print(output.get('message', ''))
    print()
    print(f"Log-likelihood: {output['opt_ll']:.4f}")
    if n_classes > 1:
        print(f"Class proportions: {np.round(output['opt_mu_prob'], 6)}")
    if robust_se:
        print("Robust standard errors, clustered on the", individual_var, "variable.")
    else:
        print("Standard errors are not robust and assume correct model specification.")
    print()

    widths = (12, 10, 11, 10, 10)
    header_top = f"{'Coefficient':^{widths[0]}} | {'Estimate':^{widths[1]}} | {'Std. Err.':^{widths[2]}} | {'[Confidence Interval]':^{widths[3] + widths[4] + 3}}"
    divider = "-+-".join("-" * w for w in widths)

    if n_common > 0:
        print("Common parameters")
        print(header_top)
        print(divider)
        for i, name in enumerate(covariate_vars[n_hetero:], start=n_hetero):
            row = " | ".join([
                f"{name:<{widths[0]}}",
                f"{B_hat[i, 0]:>{widths[1]}.6f}",
                f"{se_beta[i, 0]:>{widths[2]}.6f}",
                f"{ci_beta[i, 0, 0]:>{widths[3]}.5f}",
                f"{ci_beta[i, 0, 1]:>{widths[4]}.5f}"
            ])
            print(row)
        print()

    for c in range(n_classes):
        print(f"Class {c + 1}")
        print(f"Estimated proportion: {output['opt_mu_prob'][c]:.6f}")
        print(header_top)
        print(divider)

        for i, name in enumerate(covariate_vars[:n_hetero]):
            row = " | ".join([
                f"{name:<{widths[0]}}",
                f"{B_hat[i, c]:>{widths[1]}.6f}",
                f"{se_beta[i, c]:>{widths[2]}.6f}",
                f"{ci_beta[i, c, 0]:>{widths[3]}.5f}",
                f"{ci_beta[i, c, 1]:>{widths[4]}.5f}"
            ])
            print(row)
        print()


def _mle_estimator(df,
                   choice_var,
                   covariate_vars,
                   individual_var,
                   n_alts,
                   n_classes,
                   heterogenous_vars,
                   beta_init,
                   mu_init,
                   ci_alpha,
                   robust_se,
                   opt_method):
    """
    Estimates the multinomial logit model with discrete heterogeneity using maximum likelihood estimation,
    from initial values (beta_init, mu_init).
    """
    # Prepare data
    Y = df[[choice_var]].to_numpy()
    X = df[covariate_vars].to_numpy()
    individual_ids = df[individual_var].to_numpy()
    k = X.shape[1]

    if heterogenous_vars is None:
        n_hetero = k
    else:
        n_hetero = len(heterogenous_vars)
        if covariate_vars[:n_hetero] != list(heterogenous_vars):
            raise ValueError("heterogenous_vars must be the first entries of covariate_vars.")

    theta_init = np.concatenate([beta_init.reshape(-1, order="F"), mu_init])

    result = minimize(
        fun=_negloglik,
        x0=theta_init,
        args=(X, Y, individual_ids, n_alts, n_classes, n_hetero),
        method=opt_method,
    )

    opt_ll = -result.fun
    opt_theta = result.x
    B_hat, mu_free_hat = _unpack_theta(opt_theta, k, n_classes, n_heterogenous=n_hetero)
    mu_raw_hat = np.concatenate(([0.0], mu_free_hat)) if n_classes > 1 else np.array([0.0])
    mu_prob_hat = np.exp(mu_raw_hat - logsumexp(mu_raw_hat))

    H = approx_hess(opt_theta, _negloglik, args=(X, Y, individual_ids, n_alts, n_classes, n_hetero))
    H_inv = np.linalg.pinv(H)
    se_theta = np.sqrt(np.clip(np.diag(H_inv), 0.0, None))  # avoid negative variances from numerical issues

    z_score = norm.ppf(1 - ci_alpha / 2)
    ci_theta = [(coef - z_score * se, coef + z_score * se) for coef, se in zip(opt_theta, se_theta)]

    output = {
        'opt_ll': opt_ll,
        'opt_theta': opt_theta,
        'opt_beta': B_hat,
        'opt_mu_raw': mu_raw_hat,
        'opt_mu_prob': mu_prob_hat,
        'se_theta': se_theta,
        'ci_theta': ci_theta,
        'success': result.success,
        'message': result.message,
    }

    if robust_se:
        # J estimated by outer products of household-level scores.
        S = _score_matrix(opt_theta, X, Y, individual_ids, n_alts, n_classes, n_heterogenous=n_hetero)
        J = S.T @ S

        # Sandwich variance: H^{-1} J H^{-1}
        V_robust = H_inv @ J @ H_inv
        se_theta_r = np.sqrt(np.clip(np.diag(V_robust), 0.0, None))
        ci_r = [(coef - z_score * se, coef + z_score * se) for coef, se in zip(opt_theta, se_theta_r)]

        output['se_theta'] = se_theta_r
        output['ci_theta'] = ci_r

    return output


def _run_mle_from_start(start_idx, beta0, mu0, estimator_kwargs):
    """Run one MLE start and return its index, output, and elapsed time."""
    t_start = time.time()
    out = _mle_estimator(beta_init=beta0, mu_init=mu0, **estimator_kwargs)
    t_end = time.time()
    out['elapsed_time'] = t_end - t_start
    return start_idx, out


def estimate_MNL(df,
                 choice_var,
                 covariate_vars,
                 individual_var,
                 n_alts,
                 n_classes=1,
                 heterogenous_vars=None,
                 beta_init=None,
                 mu_init=None,
                 ci_alpha=0.05,
                 robust_se=False,
                 randomized_starts=0,
                 search_bounds=[-10, 10],
                 seed=None,
                 n_cores=1,
                 opt_method='BFGS',
                 output_log=False):
    """
    Estimates the multinomial logit model with discrete heterogeneity using maximum likelihood estimation.
    
    Optional randomized starts are drawn uniformly from the specified `search_bounds` centered around 
    (`beta_init`, `mu_init`) to help avoid local optima. Specifying `n_cores` > 1 will evaluate these 
    starts in parallel using multiprocessing.
    
    Parameters    
    ----------
    df : DataFrame
        The dataset to be used for estimation.
    choice_var : str
        The column name of the choice variable.
    covariate_vars : list
        A list of column names for the covariates.
    individual_var : str
        The column name of the individual ID variable.
        If robust standard errors are requested, standard errors are clustered by this variable.
    n_alts : int
        Number of alternatives.
    n_classes : int
        Number of latent classes. 
        If 1 (default), reduces to standard MNL.
    heterogenous_vars : list or None
        A list of column names for covariates with class-varying coefficients.
        If None (default), all covariates are assumed to have class-varying coefficients
    beta_init : array, optional
        Initial guess for coefficients. 
        If None, defaults to zeros.
    mu_init : array, optional
        Initial guess for class probabilities. 
        If None, defaults to zeros.
    ci_alpha : float
        Significance level for confidence intervals.
    robust_se : bool
        Whether to compute robust standard errors. 
        If True, standard errors are clustered by `individual_var`.
    randomized_starts : int
        Number of additional random starting points to use for optimization.
    search_bounds : list
        Bounds for random perturbations around initial values, specified as [lower_bound, upper_bound].
    seed : int or None
        Random seed for reproducibility of randomized starts.
    n_cores : int
        Number of worker processes (CPU cores) for evaluating randomized starts in parallel.
        If 1 (default), evaluates starts serially.
        Use an integer > 1 to opt in to parallel processing.
    opt_method : str
        Optimization method.
    output_log : bool
        If True, include a 'start_log' DataFrame in output showing achieved LL, elapsed time, 
        optimization status and message, starting params, parameter estimates for each start.

    Returns
    -------
    Dictionary containing:
        - optimized log-likelihood 
        - parameter estimates (coefficients and class proportions)
        - standard errors
        - confidence intervals
        - output log and results of each randomized start (if output_log=True)
    """

    # Determine number of covariate parameters to estimate
    if heterogenous_vars is not None:
        k = len(heterogenous_vars) * n_classes + (len(covariate_vars) - len(heterogenous_vars))
    else:
        k = len(covariate_vars) * n_classes

    # Base start: user-provided if available, otherwise zeros.
    if beta_init is None:
        beta_base = np.zeros(k)
    else:
        beta_base = np.asarray(beta_init).reshape(-1)

    if n_classes == 1:
        mu_base = np.array([])
    else:
        if mu_init is None:
            mu_base = np.zeros(n_classes - 1)
        else:
            mu_base = np.asarray(mu_init).reshape(-1)

    # Generate randomized starts
    rng = np.random.default_rng(seed)
    n_rand = max(int(randomized_starts)-1, 0)
    beta_perturbs = rng.uniform(low=search_bounds[0], high=search_bounds[1], size=(n_rand, k))
    beta_rand = beta_base[None, :] + beta_perturbs  # (n_rand, k)

    if n_classes > 1:
        mu_perturbs = rng.uniform(low=search_bounds[0], high=search_bounds[1], size=(n_rand, n_classes - 1))
        mu_rand = mu_base[None, :] + mu_perturbs  # (n_rand, n_classes - 1)
    else:
        mu_rand = np.empty((n_rand, 0))  # (n_rand, 0)
    
    starts = [(beta_base.copy(), mu_base.copy())] + list(zip(beta_rand, mu_rand))

    # Assign starts to workers and run optimizations
    estimator_kwargs = {
        'df': df,
        'choice_var': choice_var,
        'covariate_vars': covariate_vars,
        'individual_var': individual_var,
        'n_alts': n_alts,
        'n_classes': n_classes,
        'heterogenous_vars': heterogenous_vars,
        'ci_alpha': ci_alpha,
        'robust_se': robust_se,
        'opt_method': opt_method,
    }

    n_starts = len(starts)
    max_workers = max(1, int(n_cores))
    max_workers = min(max_workers, n_starts)

    best_output = None
    start_log = []
    start_outputs = {}

    if max_workers > 1:
        try:
            print(f"Evaluating {n_starts} randomized starts in parallel using {max_workers} cores...")
            mp_context = multiprocessing.get_context('fork')
            with ProcessPoolExecutor(max_workers=max_workers, mp_context=mp_context) as ex:
                futures = [
                    ex.submit(_run_mle_from_start, i, beta0, mu0, estimator_kwargs)
                    for i, (beta0, mu0) in enumerate(starts)
                ]
                for fut in as_completed(futures):
                    start_idx, out = fut.result()
                    if output_log:
                        start_log.append((start_idx, out['opt_ll'], out['elapsed_time'], out['success'], 
                                          out['message'], starts[start_idx][0], starts[start_idx][1],
                                          out['opt_beta'], out['opt_mu_raw'], out['opt_mu_prob']))
                        start_outputs[start_idx] = out
                    if (best_output is None) or (out['opt_ll'] > best_output['opt_ll']):
                        best_output = out
        except Exception as exc:
            print(f'Parallel starts failed ({exc}); falling back to serial evaluation.')
            for i, (beta0, mu0) in enumerate(starts):
                _, out = _run_mle_from_start(i, beta0, mu0, estimator_kwargs)
                if output_log:
                    start_log.append((i, out['opt_ll'], out['elapsed_time'], out['success'], 
                                      out['message'], starts[i][0], starts[i][1],
                                      out['opt_beta'], out['opt_mu_raw'], out['opt_mu_prob']))
                    start_outputs[i] = out
                if (best_output is None) or (out['opt_ll'] > best_output['opt_ll']):
                    best_output = out
    else:
        for i, (beta0, mu0) in enumerate(starts):
            _, out = _run_mle_from_start(i, beta0, mu0, estimator_kwargs)
            if output_log:
                start_log.append((i, out['opt_ll'], out['elapsed_time'], out['success'], 
                                  out['message'], starts[i][0], starts[i][1],
                                  out['opt_beta'], out['opt_mu_raw'], out['opt_mu_prob']))
                start_outputs[i] = out
            if (best_output is None) or (out['opt_ll'] > best_output['opt_ll']):
                best_output = out

    # Print best results
    _print_mle_output(best_output, covariate_vars, 
                     heterogenous_vars=heterogenous_vars, 
                     robust_se=robust_se, individual_var=individual_var)
    
    if output_log:
        log_df = pd.DataFrame(start_log, columns=[
            'start_idx', 'achieved_ll', 'elapsed_time', 'success', 'message', 
            'start_beta', 'start_mu', 'opt_beta', 'opt_mu_raw', 'opt_mu_prob'])
        log_df.sort_values('achieved_ll', ascending=False, inplace=True)
        best_output['start_log'] = log_df.reset_index(drop=True)
        best_output['start_outputs'] = start_outputs
    
    return best_output

### 2 latent classes, heterogeneity on brand intercepts

We estimate the sample to be 16.6% class 1 and 83.4% class 2. The two classes differ in their relative ranking of brands, where class 1 likes brand 4 the most follwed by brand 2 and class 2 likes brand 1 the most followed by brand 2. Notably the coefficient on `prev_purch` is now negative and statistically significant. We previously found a slightly positive coefficient in the homogeneous case, suggesting that there may have been spurious state dependence due to unobserved heterogeneity. The estimated value of the price and feature coefficients remain relatively robust to the inclusion of latent classes.

In [ ]:
beta_init = [2, 2, -2, 3, 
             1, 0.5, -3, -1.5, 
             0.3, -30, -0.2]
mu_init = [0]

results_2LC_int = estimate_MNL(
    df=df_long,
    choice_var='b',
    covariate_vars=['brand_1', 'brand_2', 'brand_3', 'brand_4', 'f', 'p', 'prev_purch'],
    individual_var='hh',
    n_alts=4,
    n_classes=2,
    heterogenous_vars=['brand_1', 'brand_2', 'brand_3', 'brand_4'],
    beta_init=beta_init,
    mu_init=mu_init,
    robust_se=True,
    randomized_starts=24,
    search_bounds=[-5, 5],
    n_cores=8,
    opt_method='BFGS',
    output_log=False
)

Evaluating 24 randomized starts in parallel using 8 cores...
Desired error not necessarily achieved due to precision loss.

Log-likelihood: -7923.7641
Class proportions: [0.165823 0.834177]
Robust standard errors, clustered on the hh variable.

Common parameters
Coefficient  |  Estimate  |  Std. Err.  |  [Confidence Interval] 
-------------+------------+-------------+------------+-----------
f            |   0.286216 |    0.135618 |    0.02041 |    0.55202
p            | -33.123300 |    3.874680 |  -40.71753 |  -25.52907
prev_purch   |  -0.202985 |    0.070645 |   -0.34145 |   -0.06452

Class 1
Estimated proportion: 0.165823
Coefficient  |  Estimate  |  Std. Err.  |  [Confidence Interval] 
-------------+------------+-------------+------------+-----------
brand_1      |   1.864495 |    0.822350 |    0.25272 |    3.47627
brand_2      |   2.349273 |    1.072804 |    0.24662 |    4.45193
brand_3      |  -2.124427 |    1.045901 |   -4.17436 |   -0.07450
brand_4      |   2.608090 |    0.4485

### 2 latent classes, heterogeneity on all covariates

Allowing for heterogeneity on all covariates, we find 34.7% of the sample to be of class 1 and 65.3% to be of class 2. Once again, the two classes differ in their relative ranking of brands, where class 1 likes brand 1 the most follwed by brand 4 and class 2 likes brand 2 the most followed by brand 1. Class 1 also has a less negative price coefficient and positive state dependence, whereas class 2 has a more negative price coefficient and negative state dependence. 

In [ ]:
beta_init = [2, 2, -2, 3, 0.3, -30, -0.2, 
             1, 0.5, -3, -1.5, 0.3, -30, -0.2]
mu_init = [0]

results_2LC_all = estimate_MNL(
    df=df_long,
    choice_var='b',
    covariate_vars=['brand_1', 'brand_2', 'brand_3', 'brand_4', 'f', 'p', 'prev_purch'],
    individual_var='hh',
    n_alts=4,
    n_classes=2,
    heterogenous_vars=['brand_1', 'brand_2', 'brand_3', 'brand_4', 'f', 'p', 'prev_purch'],
    beta_init=beta_init,
    mu_init=mu_init,
    robust_se=True,
    randomized_starts=24,
    search_bounds=[-5, 5],
    n_cores=8,
    opt_method='BFGS',
    output_log=False
)

Evaluating 24 randomized starts in parallel using 8 cores...
Desired error not necessarily achieved due to precision loss.

Log-likelihood: -7868.3561
Class proportions: [0.653386 0.346614]
Robust standard errors, clustered on the hh variable.

Class 1
Estimated proportion: 0.653386
Coefficient  |  Estimate  |  Std. Err.  |  [Confidence Interval] 
-------------+------------+-------------+------------+-----------
brand_1      |   1.632669 |    0.374787 |    0.89810 |    2.36724
brand_2      |   2.100684 |    0.334614 |    1.44485 |    2.75652
brand_3      |  -2.041199 |    0.335673 |   -2.69911 |   -1.38329
brand_4      |  -0.403546 |    0.426448 |   -1.23937 |    0.43228
f            |   0.151923 |    0.197502 |   -0.23517 |    0.53902
p            | -43.822258 |    3.851219 |  -51.37051 |  -36.27401
prev_purch   |  -0.259846 |    0.099731 |   -0.45532 |   -0.06438

Class 2
Estimated proportion: 0.346614
Coefficient  |  Estimate  |  Std. Err.  |  [Confidence Interval] 
-------------+--

### 4 latent classes, heterogeneity on brand intercepts

Allowing for four latent classes, there is a non-trivial proportion of the sample in each class, with the smallest class containing 9.6% of the sample. The classes are somewhat different from each other in their relative ranking of brands. The coefficient on `prev_purch` is slightly negative but not statistically significant, and the price and feature coefficients remain relatively stable.

In [ ]:
beta_init = [2, 2, -2, 3, 
             1, 0.5, -3, -1.5, 
             1, 0.5, -3, -1.5, 
             1, 0.5, -3, -1.5, 
             0.3, -30, -0.2]
mu_init = [0, 0, 0]

results_4LC_int = estimate_MNL(
    df=df_long,
    choice_var='b',
    covariate_vars=['brand_1', 'brand_2', 'brand_3', 'brand_4', 'f', 'p', 'prev_purch'],
    individual_var='hh',
    n_alts=4,
    n_classes=4,
    heterogenous_vars=['brand_1', 'brand_2', 'brand_3', 'brand_4'],
    beta_init=beta_init,
    mu_init=mu_init,
    robust_se=True,
    randomized_starts=24,
    search_bounds=[-5, 5],
    n_cores=8,
    opt_method='BFGS',
    output_log=False
)

Evaluating 24 randomized starts in parallel using 8 cores...
Desired error not necessarily achieved due to precision loss.

Log-likelihood: -7090.0323
Class proportions: [0.095589 0.421946 0.099706 0.382759]
Robust standard errors, clustered on the hh variable.

Common parameters
Coefficient  |  Estimate  |  Std. Err.  |  [Confidence Interval] 
-------------+------------+-------------+------------+-----------
f            |   0.397243 |    0.130088 |    0.14228 |    0.65221
p            | -35.562669 |    3.557772 |  -42.53577 |  -28.58956
prev_purch   |  -0.116616 |    0.063262 |   -0.24061 |    0.00738

Class 1
Estimated proportion: 0.095589
Coefficient  |  Estimate  |  Std. Err.  |  [Confidence Interval] 
-------------+------------+-------------+------------+-----------
brand_1      |   2.072322 |    0.609743 |    0.87725 |    3.26740
brand_2      |   0.015900 |    0.584692 |   -1.13008 |    1.16188
brand_3      |  -4.326619 |    1.059497 |   -6.40319 |   -2.25004
brand_4      |   2.

## Compute elasticities

We seek to estimate the cross-price elasticity of brand $j$'s market share to a change in brand $k$'s price. For simplicity, I explain how to do this for the homogeneous case (i.e., no latent classes). First, note that given a set of parameters $\beta$ and design matrix $X$, we can predict each brand's market share by computing the choice probabilities $P_{ijt}$ of each brand $j$ for each household-trip choice occasion $it$ using the multinomial logit formula. The market share for brand $j$ is given by
$$
m_j = \frac{1}{NT} \sum_{i=1}^N \sum_{t=1}^T P_{ijt}
$$
Incorporating latent choices simply involves computing class-specific shares $m_{j\mid c}$ and then integrating over class proportions $\mu$. The `predict_market_shares` function implements these computations and returns market shares given $\beta, \mu, X$. 

Now we can simulate a small $\delta\%$ change in the price of brand $k$ by multiplying each observed price for brand $k$ by $1 + 0.01\delta$ to create $\tilde{X}$. Providing estimated parameters $\beta$ and $\tilde X$ to `predict_market_shares` gives us each brand's market shares under these perturbed prices. Note that only brand $k$'s price changes in this counterfactual (all other prices and features are held constant). Elasticity is then given by
$$
e_{jk} = \frac{\partial \log s_j}{\partial \log p_k} \approx \frac{\Delta s_j / s_j}{\Delta p_k / p_k }
$$
In the implementation, I employ a central differencing approach that computes the share $s_j^+$ under a $\delta\%$ increase in $p_k$ and the share $s_j^-$ under a $\delta\%$ decrease in $p_k$. Then, 
$$
e_{jk} \approx \frac{\Delta s_j / s_j}{\Delta p_k / p_k } = \frac{s_j^+ - s_j^-}{2\delta \cdot s_j^0}
$$
where $s_j^0$ is the share at original prices $p_k$.

In [8]:
def predict_market_shares(results, df, covariate_vars, n_alts):
    """
    Predict market shares for each brand under an MNL/finite-mixture MNL model.
    """
    X = df[covariate_vars].to_numpy()
    B_hat = np.asarray(results['opt_beta'])
    mu_raw = np.asarray(results['opt_mu_raw'], dtype=float).reshape(-1)
    if B_hat.ndim == 1:
        B_hat = B_hat[:, None]

    n_choices = X.shape[0] // n_alts
    n_classes = B_hat.shape[1]

    # Compute choice probabilities according to MNL with outside option
    V = (X @ B_hat).reshape(n_choices, n_alts, n_classes, order='C')
    V0 = np.concatenate([np.zeros((n_choices, 1, n_classes)), V], axis=1)
    log_denom = logsumexp(V0, axis=1)
    log_Pr = V - log_denom[:, None, :]  # (NT, J, C)
    log_shares_by_class = logsumexp(log_Pr, axis=0) - np.log(n_choices)  # (J, C)
    log_mu = mu_raw - logsumexp(mu_raw)
    log_shares = logsumexp(log_shares_by_class + log_mu[None, :], axis=1)  # (J,)
    return np.exp(log_shares)


def cross_price_elasticity_matrix(
    results,
    df,
    covariate_vars,
    n_alts,
    price_var='p',
    delta=0.01,
    alt_col='brand_num',
):
    """
    Numerically compute cross-price elasticities e_{jk} = d ln(s_j) / d ln(p_k) by perturbing each brand's price 
    and predicting new shares using the estimated model. 

    Outputs an n_alts x n_alts DataFrame where entry (j, k) is the elasticity of brand j's share with respect to brand k's price.
    """
    elas_mat = np.zeros((n_alts, n_alts), dtype=float)
    alt_idx = df[alt_col].to_numpy()
    base_shares = predict_market_shares(results, df, covariate_vars, n_alts)

    for k in range(1, n_alts + 1):
        shock_mask = alt_idx == k

        df_plus = df.copy()
        df_minus = df.copy()
        df_plus.loc[shock_mask, price_var] = df_plus.loc[shock_mask, price_var] * (1.0 + delta)
        df_minus.loc[shock_mask, price_var] = df_minus.loc[shock_mask, price_var] * (1.0 - delta)

        plus_shares = predict_market_shares(results, df_plus, covariate_vars, n_alts)
        minus_shares = predict_market_shares(results, df_minus, covariate_vars, n_alts)
        elas_mat[:, k - 1] = (plus_shares - minus_shares) / (2.0 * delta * base_shares) 

        row_names = [f'share_{j}' for j in range(1, n_alts + 1)]
        col_names = [f'price_{k}' for k in range(1, n_alts + 1)]
    return pd.DataFrame(elas_mat, index=row_names, columns=col_names)

In [9]:
# Homogenous MNL
results_homog = estimate_MNL(
    df=df_long,
    choice_var='b',
    covariate_vars=['brand_1', 'brand_2', 'brand_3', 'brand_4', 'f', 'p', 'prev_purch'],
    individual_var='hh',
    n_alts=4,
    robust_se=True,
    opt_method='BFGS',
    output_log=True
)

elas_homog = cross_price_elasticity_matrix(
    results=results_homog, 
    df=df_long, 
    covariate_vars=['brand_1', 'brand_2', 'brand_3', 'brand_4', 'f', 'p', 'prev_purch'], 
    n_alts=4, 
    price_var='p', 
    delta=0.01,
    alt_col='brand_num'
)
elas_homog

Desired error not necessarily achieved due to precision loss.

Log-likelihood: -8759.3930
Robust standard errors, clustered on the hh variable.

Class 1
Estimated proportion: 1.000000
Coefficient  |  Estimate  |  Std. Err.  |  [Confidence Interval] 
-------------+------------+-------------+------------+-----------
brand_1      |   0.687177 |    0.427138 |   -0.15000 |    1.52435
brand_2      |   0.079288 |    0.447725 |   -0.79824 |    0.95681
brand_3      |  -3.596405 |    0.418539 |   -4.41673 |   -2.77608
brand_4      |  -0.581285 |    0.267113 |   -1.10482 |   -0.05775
f            |   0.255392 |    0.128754 |    0.00304 |    0.50775
p            | -33.175453 |    3.423087 |  -39.88458 |  -26.46633
prev_purch   |   0.146533 |    0.113504 |   -0.07593 |    0.36900



,price_1,price_2,price_3,price_4
share_1,-3.022004,0.194570,0.008810,0.110601
share_2,0.210052,-2.430456,0.009132,0.113768
share_3,0.213018,0.204327,-1.638967,0.114236
share_4,0.215535,0.205952,0.009241,-2.479743


In [10]:
elas_2LC_int = cross_price_elasticity_matrix(
    results=results_2LC_int, 
    df=df_long, 
    covariate_vars=['brand_1', 'brand_2', 'brand_3', 'brand_4', 'f', 'p', 'prev_purch'], 
    n_alts=4, 
    price_var='p', 
    delta=0.01,
    alt_col='brand_num'
)
elas_2LC_int

,price_1,price_2,price_3,price_4
share_1,-2.993237,0.206020,0.008933,0.134716
share_2,0.202103,-2.329669,0.009724,0.314545
share_3,0.210233,0.234169,-1.636419,0.174954
share_4,0.196658,0.474474,0.010792,-1.973632


The estimated elasticities are mostly similar between the two models, with noticeable differences being in $4,2$ and $2,4$. As expected, the own-price elasticities are negative, confirming the law of demand. The cross-price elasticities are positive, suggesting that the products are substitutes. Moreover (especially in the homogenous model), the cross-price elasticities within a column are quite similar---a feature of IIA. The elasticities associated with a change in the price of good 1 are the highest, suggesting that good 1 has more influence on the market. On the other hand, changes in the price of good 3 move market shares very little. The main difference between the estimated elasticities of the two models is the strong substitution pattern observed between goods 2 and 4 in the 2-latent-class model that does not appear in the homogenous model.

## Simulations

In this section, I simulate synthetic data to verify that the `estimate_mle` function can correctly uncover the underlying parameters. I also test the behavior of the function when an unnecessary latent class is estimated (i.e., the model is not point-identified).

In [11]:
# Simulate synthetic data generated accorinding to MNL with 3 latent classes
from itertools import permutations

np.random.seed(94305)

# Problem dimensions
n_classes = 3
n_alts = 4
n_covariates = 5
n_hetero = 3
n_common = n_covariates - n_hetero
n_households = 500
n_choices_per_hh = 20
n_choices = n_households * n_choices_per_hh

# True parameters
B_true = np.zeros((n_covariates, n_classes))
B_true[:n_hetero, 0] = [2.0, -0.5, 0.5]
B_true[:n_hetero, 1] = [-1.0, -1.5, 1.5]
B_true[:n_hetero, 2] = [0.5, 1.0, -1.0]
B_true[n_hetero:, :] = [[-2.0], [0.8]]  # common coefficients
mu_true = np.array([0.2, 0.5, 0.3])

# Assign households to latent classes
hh_ids = np.arange(n_households)
n_c = (mu_true * n_households).astype(int)
class_by_hh = np.empty(n_households, dtype=int)
class_by_hh[:n_c[0]] = 0
class_by_hh[n_c[0]:n_c[0] + n_c[1]] = 1
class_by_hh[n_c[0] + n_c[1]:] = 2

# Simulate each choice-occasion
hh_by_choice = np.repeat(hh_ids, n_choices_per_hh)
choice_id = np.arange(n_choices)
class_by_choice = class_by_hh[hh_by_choice]
X_all = np.random.normal(size=(n_choices, n_alts, n_covariates))
B_by_choice = B_true[:, class_by_choice].T  # (NT, K)
V_inside = np.einsum("ntk,nk->nt", X_all, B_by_choice)  # (NT, J)
V0 = np.concatenate([np.zeros((n_choices, 1)), V_inside], axis=1)
probs = np.exp(V0 - logsumexp(V0, axis=1, keepdims=True))
u = np.random.rand(n_choices, 1)
cumprobs = np.cumsum(probs, axis=1)
choices = (u > cumprobs).sum(axis=1)

# Build dataset (long format)
y_inside = (choices[:, None] == np.arange(1, n_alts + 1)).astype(int)
alt = np.tile(np.arange(1, n_alts + 1), n_choices)
x_flat = X_all.reshape(n_choices * n_alts, n_covariates)
df_sim = pd.DataFrame({
    "hh": np.repeat(hh_by_choice, n_alts),
    "choice_id": np.repeat(choice_id, n_alts),
    "alt": alt,
    "y": y_inside.reshape(-1),
    "x1": x_flat[:, 0],
    "x2": x_flat[:, 1],
    "x3": x_flat[:, 2],
    "x4": x_flat[:, 3],
    "x5": x_flat[:, 4],
})

In [12]:
# Print true parameters for reference
print("True B parameters:")
print("Common coefficients:", B_true[n_hetero:, 0])
for c in range(n_classes):
    print(f"Class {c + 1}: {B_true[:n_hetero, c]}")
print(f"True class probabilities: {mu_true}")

# Estimate
sim_results = estimate_MNL(
    df=df_sim,
    choice_var="y",
    covariate_vars=["x1", "x2", "x3", "x4", "x5"],
    individual_var="hh",
    n_alts=n_alts,
    n_classes=n_classes,
    heterogenous_vars=["x1", "x2", "x3"],
    robust_se=True,
    randomized_starts=24,
    search_bounds=[-5, 5],
    n_cores=8,
    opt_method="BFGS",
    output_log=True
)

True B parameters:
Common coefficients: [-2.   0.8]
Class 1: [ 2.  -0.5  0.5]
Class 2: [-1.  -1.5  1.5]
Class 3: [ 0.5  1.  -1. ]
True class probabilities: [0.2 0.5 0.3]
Evaluating 24 randomized starts in parallel using 8 cores...
Desired error not necessarily achieved due to precision loss.

Log-likelihood: -7658.0457
Class proportions: [0.200011 0.499999 0.29999 ]
Robust standard errors, clustered on the hh variable.

Common parameters
Coefficient  |  Estimate  |  Std. Err.  |  [Confidence Interval] 
-------------+------------+-------------+------------+-----------
x4           |  -2.007901 |    0.025841 |   -2.05855 |   -1.95725
x5           |   0.818937 |    0.019327 |    0.78106 |    0.85682

Class 1
Estimated proportion: 0.200011
Coefficient  |  Estimate  |  Std. Err.  |  [Confidence Interval] 
-------------+------------+-------------+------------+-----------
x1           |   2.001888 |    0.048357 |    1.90711 |    2.09666
x2           |  -0.525658 |    0.043622 |   -0.61116 |  

In [22]:
# Behavior when an unnecessary class is added

# Print true parameters for reference
print("True B parameters:")
print("Common coefficients:", B_true[n_hetero:, 0])
for c in range(n_classes):
    print(f"Class {c + 1}: {B_true[:n_hetero, c]}")
print(f"True class probabilities: {mu_true}")

# Estimate
sim_results = estimate_MNL(
    df=df_sim,
    choice_var="y",
    covariate_vars=["x1", "x2", "x3", "x4", "x5"],
    individual_var="hh",
    n_alts=n_alts,
    n_classes=n_classes + 1,
    heterogenous_vars=["x1", "x2", "x3"],
    robust_se=True,
    randomized_starts=24,
    search_bounds=[-5, 5],
    n_cores=8,
    opt_method="BFGS",
    output_log=True
)

True B parameters:
Common coefficients: [-2.   0.8]
Class 1: [ 2.  -0.5  0.5]
Class 2: [-1.  -1.5  1.5]
Class 3: [ 0.5  1.  -1. ]
True class probabilities: [0.2 0.5 0.3]
Evaluating 24 randomized starts in parallel using 8 cores...
Desired error not necessarily achieved due to precision loss.

Log-likelihood: -7656.9063
Class proportions: [0.447563 0.29999  0.200011 0.052437]
Robust standard errors, clustered on the hh variable.

Common parameters
Coefficient  |  Estimate  |  Std. Err.  |  [Confidence Interval] 
-------------+------------+-------------+------------+-----------
x4           |  -2.012404 |    0.026315 |   -2.06398 |   -1.96083
x5           |   0.820979 |    0.019361 |    0.78303 |    0.85893

Class 1
Estimated proportion: 0.447563
Coefficient  |  Estimate  |  Std. Err.  |  [Confidence Interval] 
-------------+------------+-------------+------------+-----------
x1           |  -0.992546 |    0.046985 |   -1.08463 |   -0.90046
x2           |  -1.472776 |    0.035279 |   -1.

In [24]:
log = sim_results['start_log']
log_outputs = sim_results['start_outputs']
log

,start_idx,achieved_ll,elapsed_time,success,message,start_beta,start_mu,opt_beta,opt_mu_raw,opt_mu_prob
0,10,-7656.906338,11.836884,False,Desired error not necessarily achieved due to ...,"[-1.0437303544517307, -2.1070201713486245, -4....","[-0.6380886276105846, 2.5854283635910646, -0.3...","[[-0.9925461753511852, 0.5034491386749287, 2.0...","[0.0, -0.4000687688650681, -0.8054464382011721...","[0.4475629256914433, 0.29998977028793333, 0.20..."
1,7,-7658.045725,9.888670,False,Desired error not necessarily achieved due to ...,"[0.2582863984222117, -3.3619268302551744, 4.17...","[-4.264204845088251, -1.2377893862093658, -4.6...","[[-1.0363543878018178, 1.483452447360663, 0.50...","[0.0, -26.271002300642177, -0.5108563800487963...","[0.49999881809005187, 1.948128700202536e-12, 0..."
2,19,-7658.045725,11.863806,False,Desired error not necessarily achieved due to ...,"[-1.1003923401682958, -0.7469801180469438, 1.3...","[2.2409489254232295, -0.27353708625007966, -0....","[[-1.0363543637856152, 46.97104007433035, 0.50...","[0.0, -20.46498998518867, -0.5108566283704892,...","[0.4999988929336866, 6.473465234421833e-10, 0...."
3,12,-7658.045744,19.703654,False,Desired error not necessarily achieved due to ...,"[-3.761503619782476, 4.386445819746582, 2.2122...","[-2.0157656756118314, -0.8657866746250642, 4.4...","[[-13.229494277265458, -1.036370405564459, 0.5...","[0.0, 16.577369854660866, 16.066528067672486, ...","[3.1586525199565066e-08, 0.4999909051777605, 0..."
4,0,-7658.045745,16.728401,False,Desired error not necessarily achieved due to ...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0]","[[2.0018880205559304, 0.5027379704656275, -1.6...","[0.0, 0.40537606698808176, -15.407680312461352...","[0.20001116289587403, 0.29999003170613037, 4.0..."
5,3,-7658.045748,13.567670,False,Desired error not necessarily achieved due to ...,"[-3.3453227419413514, 2.5486634345829584, 1.38...","[-1.028354338949895, -4.6893132896418175, -0.2...","[[-1.0363544253209094, 2.0018877978387875, 4.9...","[0.0, -0.9162322088089003, -16.203199828965552...","[0.4999987227723229, 0.20001119403458195, 4.59..."
6,17,-7658.045750,11.846038,False,Desired error not necessarily achieved due to ...,"[-4.269504046994807, 0.32427033784962145, 1.30...","[2.7623825451646855, -0.7840003083107738, 4.34...","[[2.00188789280839, 10.127136869069933, -1.036...","[0.0, -15.188492572886501, 0.9162339196262466,...","[0.20001091233520785, 5.0672832143284814e-08, ..."
7,23,-7658.045761,11.987385,False,Desired error not necessarily achieved due to ...,"[1.871725998837701, 1.962296248904526, 4.36894...","[-4.433462828693387, 0.9173070014317641, 0.131...","[[-1.036354454604164, 2.001888134920464, 5.009...","[0.0, -0.9162330995220425, -15.745589763354984...","[0.49999878272729453, 0.200011039865433, 7.256..."
8,22,-7658.045803,16.289673,False,Desired error not necessarily achieved due to ...,"[-4.7822159448473105, 3.9581059877296543, -2.5...","[-0.06413168497050648, 1.770650639777549, -4.3...","[[0.5027382683910668, 2.0018883438508897, -1.0...","[0.0, -0.4053760557669591, 0.510853838200131, ...","[0.2999903777158709, 0.20001139583392705, 0.49..."
9,4,-7658.045831,13.100842,False,Desired error not necessarily achieved due to ...,"[-1.7007590122951255, -2.24865035480215, 3.250...","[-3.712739107230477, -0.1502384914798005, -0.8...","[[55.11956467042078, -1.0363544182081768, 2.00...","[0.0, 14.670926793655266, 13.754693293493556, ...","[2.1255262282066025e-07, 0.49999886589100934, ..."


In [25]:
_print_mle_output(log_outputs[7], covariate_vars=["x1", "x2", "x3", "x4", "x5"], heterogenous_vars=["x1", "x2", "x3"], robust_se=True, individual_var="hh")

Desired error not necessarily achieved due to precision loss.

Log-likelihood: -7658.0457
Class proportions: [0.499999 0.       0.29999  0.200011]
Robust standard errors, clustered on the hh variable.

Common parameters
Coefficient  |  Estimate  |  Std. Err.  |  [Confidence Interval] 
-------------+------------+-------------+------------+-----------
x4           |  -2.007901 |    0.025841 |   -2.05855 |   -1.95725
x5           |   0.818937 |    0.019327 |    0.78106 |    0.85682

Class 1
Estimated proportion: 0.499999
Coefficient  |  Estimate  |  Std. Err.  |  [Confidence Interval] 
-------------+------------+-------------+------------+-----------
x1           |  -1.036354 |    0.029076 |   -1.09334 |   -0.97937
x2           |  -1.468371 |    0.030840 |   -1.52882 |   -1.40792
x3           |   1.431456 |    0.029604 |    1.37343 |    1.48948

Class 2
Estimated proportion: 0.000000
Coefficient  |  Estimate  |  Std. Err.  |  [Confidence Interval] 
-------------+------------+-------------